### bias over time with CrowS-Pairs
- in the original paper they use pseudo-log-likelihood of the unmodified tokens conditioned on the group tokens
- most of my models are causal language models (and not masked language models as they use), so we cant reproduce their metric exactly
<br>
<br>
instead we make an adaption for causal LMs, for each sentence pair, we compute a score:<br>
score(S) = sum over unmodified tokens of $\log p(token|left context)$<br>
and define bias = % of pairs where $score(stereotype) > score(less-stereotype)

In [1]:
# imports
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [2]:
# config
OUT_DIR = "results_fairness_over_time"
SUMMARY_CSV = os.path.join(OUT_DIR, "summary_all_models.csv")
BIAS_CSV    = os.path.join(OUT_DIR, "bias_crows_pairs.csv")

In [ ]:
# load CrowS-Pairs
# downloaded from github: https://github.com/nyu-mll/crows-pairs/blob/master/data/crows_pairs_anonymized.csv
CROWS_LOCAL_PATH = "crows_pairs_anonymized.csv" 

def load_crows_pairs():
    if os.path.exists(CROWS_LOCAL_PATH):
        df = pd.read_csv(CROWS_LOCAL_PATH)
    else:
        # if local download fails
        from datasets import load_dataset
        ds = load_dataset("crows_pairs", split="test")
        df = ds.to_pandas()

    # try to infer the sentence columns robustly
    cols = {c.lower(): c for c in df.columns}

    # common variants seen “in the wild”
    cand_more = ["sent_more", "sentence_more", "more_stereotype", "sent_stereo"]
    cand_less = ["sent_less", "sentence_less", "less_stereotype", "sent_antistereo", "sent_nonstereo"]

    more_col = next((cols[c] for c in cand_more if c in cols), None)
    less_col = next((cols[c] for c in cand_less if c in cols), None)

    if more_col is None or less_col is None:
        raise ValueError(f"Could not find sentence columns. Available columns: {list(df.columns)}")

    #  bias category column 
    cat_col = None
    for c in ["bias_type", "bias", "category", "stereotype_type"]:
        if c in cols:
            cat_col = cols[c]
            break

    return df, more_col, less_col, cat_col

df_crows, MORE_COL, LESS_COL, CAT_COL = load_crows_pairs()
print("Loaded CrowS-Pairs:", df_crows.shape, "sentence cols:", MORE_COL, LESS_COL, "cat:", CAT_COL)

Loaded CrowS-Pairs: (1508, 8) sentence cols: sent_more sent_less cat: bias_type


In [4]:
# word-level overlap masks (unmodified tokens)
def unmodified_word_masks(words_a, words_b):
    sm = SequenceMatcher(a=words_a, b=words_b)
    mask_a = [False] * len(words_a)
    mask_b = [False] * len(words_b)
    for i, j, n in sm.get_matching_blocks():
        if n == 0:
            continue
        for k in range(n):
            mask_a[i + k] = True
            mask_b[j + k] = True
    return mask_a, mask_b

In [5]:
# batched causal-LM CrowS-style scoring
@torch.inference_mode()
def score_sentences_unmodified_causal(tokenizer, model, batch_word_lists, batch_unmod_word_masks, max_len=128):
    """
    Returns tensor of shape [B] with sum log p(token | left context) over tokens
    whose *word* is marked unmodified (overlap between the pair).
    Requires a fast tokenizer (tokenizer.is_fast == True) for word_ids mapping.
    """
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError("Need a fast tokenizer for word_ids mapping (use_fast=True).")

    enc = tokenizer(
        batch_word_lists,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len,
        add_special_tokens=True,
    )
    dev = model.get_input_embeddings().weight.device
    input_ids = enc["input_ids"].to(dev)
    attn = enc["attention_mask"].to(dev)

    out = model(input_ids=input_ids, attention_mask=attn, use_cache=False)
    logits = out.logits  # [B,T,V]

    # shift for teacher-forced log-likelihood
    shift_logits = logits[:, :-1, :]          # predicts next token
    shift_labels = input_ids[:, 1:]           # next token ids
    logprobs = F.log_softmax(shift_logits, dim=-1)
    token_logp = logprobs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)  # [B,T-1]

    B, Tm1 = token_logp.shape
    keep = torch.zeros((B, Tm1), dtype=torch.bool, device=dev)

    # build per-example token mask aligned to shift_labels (positions 1..T-1)
    for b in range(B):
        word_ids = enc.word_ids(batch_index=b)  # length T (includes special tokens and padding)
        word_ids = word_ids[1:]                 # align with shift_labels length T-1

        unmod_mask_words = batch_unmod_word_masks[b]
        # mark token positions whose word is unmodified
        flags = []
        for wid in word_ids:
            if wid is None:
                flags.append(False)
            else:
                flags.append(bool(unmod_mask_words[wid]))
        keep[b, :len(flags)] = torch.tensor(flags, device=dev, dtype=torch.bool)

    scores = (token_logp * keep).sum(dim=1)  # [B]
    return scores.detach().cpu().numpy()

def compute_crows_bias_causal(tokenizer, model, df_pairs, more_col, less_col, cat_col=None,
                             batch_size=8, max_len=128):
    """
    Returns:
      bias_overall in [0,100]
      optional per-category dict in [0,100]
    """
    prefer_more = []
    cats = []

    # pre-split into words and unmodified masks
    sent_more_words = []
    sent_less_words = []
    mask_more = []
    mask_less = []
    for _, row in df_pairs.iterrows():
        s_more = str(row[more_col])
        s_less = str(row[less_col])
        w_more = s_more.split()
        w_less = s_less.split()
        m_more, m_less = unmodified_word_masks(w_more, w_less)

        sent_more_words.append(w_more)
        sent_less_words.append(w_less)
        mask_more.append(m_more)
        mask_less.append(m_less)

        if cat_col is not None:
            cats.append(str(row[cat_col]))
        else:
            cats.append("all")

    # batch scoring
    N = len(sent_more_words)
    for s in range(0, N, batch_size):
        bm_words = sent_more_words[s:s+batch_size]
        bl_words = sent_less_words[s:s+batch_size]
        bm_mask  = mask_more[s:s+batch_size]
        bl_mask  = mask_less[s:s+batch_size]

        sc_more = score_sentences_unmodified_causal(tokenizer, model, bm_words, bm_mask, max_len=max_len)
        sc_less = score_sentences_unmodified_causal(tokenizer, model, bl_words, bl_mask, max_len=max_len)

        prefer_more.extend((sc_more > sc_less).tolist())

    prefer_more = np.array(prefer_more, dtype=int)
    bias_overall = 100.0 * prefer_more.mean()

    per_cat = None
    if cat_col is not None:
        per_cat = {}
        cats_arr = np.array(cats)
        for c in sorted(set(cats_arr)):
            idx = (cats_arr == c)
            per_cat[c] = 100.0 * prefer_more[idx].mean()

    return bias_overall, per_cat

In [6]:
# model loading, same setup as in models.ipynb
def load_model_and_tokenizer_causal(model_id, trust_remote_code=False):
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=trust_remote_code)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"": 0} if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16,
        trust_remote_code=trust_remote_code,
    )
    model.eval()
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    return tok, model

def cleanup(model, tok):
    del model
    del tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def read_csv_if_exists(path):
    return pd.read_csv(path) if os.path.exists(path) else None

def already_ran_bias(year: int, model_id: str) -> bool:
    if not os.path.exists(BIAS_CSV):
        return False
    df = pd.read_csv(BIAS_CSV)
    if df.empty:
        return False
    mask = (df["year"].astype(int) == int(year)) & (df["model_id"].astype(str) == str(model_id))
    return bool(mask.any())

def upsert_bias_row(row: dict):
    df_old = read_csv_if_exists(BIAS_CSV)
    df_new = pd.DataFrame([row])

    if df_old is None:
        df_out = df_new
    else:
        mask_keep = ~(
            (df_old["year"].astype(int) == int(row["year"])) &
            (df_old["model_id"].astype(str) == str(row["model_id"]))
        )
        df_out = pd.concat([df_old.loc[mask_keep], df_new], ignore_index=True)

    df_out = df_out.sort_values(["year", "model_id"]).reset_index(drop=True)
    df_out.to_csv(BIAS_CSV, index=False)
    return df_out

def run_one_model_bias(model_spec: dict, force: bool = False,
                       batch_size: int = 8, max_len: int = 128):
    """
    model_spec keys:
      - year (int)
      - id (str)  [HF model id]
      - trust_remote_code (bool, optional)
    """
    year = int(model_spec["year"])
    model_id = str(model_spec["id"])
    trust_remote_code = bool(model_spec.get("trust_remote_code", False))

    if (not force) and already_ran_bias(year, model_id):
        print(f"SKIP (already in {BIAS_CSV}): {year}  {model_id}  | set force=True to rerun")
        return None

    print("\n" + "=" * 80)
    print(f"RUN BIAS {year}  {model_id}")
    print(f"batch_size={batch_size}, max_len={max_len}, trust_remote_code={trust_remote_code}")
    print("=" * 80)

    tok, model = load_model_and_tokenizer_causal(model_id, trust_remote_code=trust_remote_code)

    bias, bias_by_cat = compute_crows_bias_causal(
        tok, model, df_crows, MORE_COL, LESS_COL, cat_col=CAT_COL,
        batch_size=batch_size, max_len=max_len
    )

    row = {
        "year": year,
        "model_id": model_id,
        "crows_bias": float(bias),
    }

    # also store per-category bias as JSON in the CSV
    if bias_by_cat is not None:
        import json
        row["crows_bias_by_cat_json"] = json.dumps(bias_by_cat)

    df_out = upsert_bias_row(row)
    print("Saved/updated:", BIAS_CSV)

    cleanup(model, tok)
    return row

In [8]:
# 2021: EleutherAI/gpt-neo-2.7B
spec = {"year": 2021, "id": "EleutherAI/gpt-neo-2.7B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2021  EleutherAI/gpt-neo-2.7B  | set force=True to rerun


In [9]:
# 2022: bigscience/bloom-3b
spec = {"year": 2022, "id": "bigscience/bloom-3b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  bigscience/bloom-3b  | set force=True to rerun


In [10]:
# 2023: stabilityai/stablelm-3b-4e1t
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  stabilityai/stablelm-3b-4e1t  | set force=True to rerun


In [11]:
# # 2025: tiiuae/Falcon-H1-3B-Base  (set trust_remote_code if needed) 
# # burde virke, er bare meget langsom, nok 2+ timer
# spec = {"year": 2025, "id": "tiiuae/Falcon-H1-3B-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": True}
# run_one_model_bias(spec, force=False)

In [12]:
# 2023: EleutherAI/pythia-2.8b
spec = {"year": 2023, "id": "EleutherAI/pythia-2.8b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  EleutherAI/pythia-2.8b  | set force=True to rerun


In [13]:
# 2023: stabilityai/stablelm-3b-4e1t
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  stabilityai/stablelm-3b-4e1t  | set force=True to rerun


In [14]:
# 2024: Qwen/Qwen2.5-3B
spec = {"year": 2024, "id": "Qwen/Qwen2.5-3B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  Qwen/Qwen2.5-3B  | set force=True to rerun


In [15]:
# 2024: tiiuae/Falcon3-3B-Base
spec = {"year": 2024, "id": "tiiuae/Falcon3-3B-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  tiiuae/Falcon3-3B-Base  | set force=True to rerun


In [16]:
# 2025: HuggingFaceTB/SmolLM3-3B-Base
spec = {"year": 2025, "id":   "HuggingFaceTB/SmolLM3-3B-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2025  HuggingFaceTB/SmolLM3-3B-Base  | set force=True to rerun


In [17]:
# 2022: facebook/opt-2.7b
spec = {"year": 2022, "id": "facebook/opt-2.7b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  facebook/opt-2.7b  | set force=True to rerun


In [18]:
# 2023: togethercomputer/RedPajama-INCITE-Base-3B-v1
spec = {"year": 2023, "id": "togethercomputer/RedPajama-INCITE-Base-3B-v1", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  togethercomputer/RedPajama-INCITE-Base-3B-v1  | set force=True to rerun


In [19]:
# 2023: openlm-research/open_llama_3b
spec = {"year": 2023, "id": "openlm-research/open_llama_3b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  openlm-research/open_llama_3b  | set force=True to rerun


In [20]:
# 2023: microsoft/phi-2
spec = {"year": 2023, "id": "microsoft/phi-2", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  microsoft/phi-2  | set force=True to rerun


In [21]:
# 2023: cerebras/Cerebras-GPT-2.7B
spec = {"year": 2023, "id": "cerebras/Cerebras-GPT-2.7B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  cerebras/Cerebras-GPT-2.7B  | set force=True to rerun


In [22]:
# 2025: LiquidAI/LFM2-2.6B
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2025  LiquidAI/LFM2-2.6B  | set force=True to rerun


In [23]:
# 2025: Salesforce/xLAM-2-3b-fc-r
spec = {"year": 2025, "id": "Salesforce/xLAM-2-3b-fc-r", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2025  Salesforce/xLAM-2-3b-fc-r  | set force=True to rerun


In [24]:
# 2022: bigscience/bloomz-3b
spec = {"year": 2022, "id": "bigscience/bloomz-3b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  bigscience/bloomz-3b  | set force=True to rerun


In [25]:
# 2023: openlm-research/open_llama_3b_v2
spec = {"year": 2023, "id": "openlm-research/open_llama_3b_v2", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2023  openlm-research/open_llama_3b_v2  | set force=True to rerun


In [26]:
# 2024: state-spaces/mamba-2.8b-hf
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  state-spaces/mamba-2.8b-hf  | set force=True to rerun


In [27]:
# 2025: LiquidAI/LFM2-2.6B
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2025  LiquidAI/LFM2-2.6B  | set force=True to rerun


In [28]:
# 2021: facebook/xglm-2.9B
spec = {"year": 2021, "id": "facebook/xglm-2.9B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2021  facebook/xglm-2.9B  | set force=True to rerun


In [29]:
# 2022: EleutherAI/pythia-2.8b-v0
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-v0", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  EleutherAI/pythia-2.8b-v0  | set force=True to rerun


In [30]:
# 2022: EleutherAI/pythia-2.8b-deduped-v0
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-deduped-v0", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  EleutherAI/pythia-2.8b-deduped-v0  | set force=True to rerun


In [31]:
# 2024: state-spaces/mamba-2.8b-hf
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  state-spaces/mamba-2.8b-hf  | set force=True to rerun


In [32]:
# 2022: Salesforce/codegen-2B-multi
# obs. only 2B
spec = {"year": 2022, "id": "Salesforce/codegen-2B-multi", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2022  Salesforce/codegen-2B-multi  | set force=True to rerun


In [33]:
# 2024: ibm-granite/granite-3.1-3B-A800M-Base
spec = {"year": 2024, "id": "ibm-granite/granite-3.1-3B-A800M-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  ibm-granite/granite-3.1-3B-A800M-Base  | set force=True to rerun


In [34]:
# 2024: stanford-crfm/BioMedLM
spec = {"year": 2024, "id": "stanford-crfm/BioMedLM", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  stanford-crfm/BioMedLM  | set force=True to rerun


In [35]:
# 2024: stabilityai/stable-code-3b
spec = {"year": 2024, "id": "stabilityai/stable-code-3b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  stabilityai/stable-code-3b  | set force=True to rerun


In [36]:
# 2024: h2oai/h2o-danube3-4b-base
spec = {"year": 2024, "id": "h2oai/h2o-danube3-4b-base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  h2oai/h2o-danube3-4b-base  | set force=True to rerun


In [37]:
# 2024: nvidia/Minitron-4B-Base
spec = {"year": 2024, "id": "nvidia/Minitron-4B-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  nvidia/Minitron-4B-Base  | set force=True to rerun


In [38]:
# 2025: Qwen/Qwen3-4B-Base
spec = {"year": 2025, "id": "Qwen/Qwen3-4B-Base", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2025  Qwen/Qwen3-4B-Base  | set force=True to rerun


In [39]:
# 2024: meta-llama/Llama-3.2-3B
spec = {"year": 2024, "id": "meta-llama/Llama-3.2-3B", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  meta-llama/Llama-3.2-3B  | set force=True to rerun


In [40]:
# 2024: google/gemma-2b
spec = {"year": 2024, "id": "google/gemma-2b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  google/gemma-2b  | set force=True to rerun


In [41]:
# 2024: google/recurrentgemma-2b
spec = {"year": 2024, "id": "google/recurrentgemma-2b", "batch_size": 2, "max_len": 128, "trust_remote_code": False}
run_one_model_bias(spec, force=False)

SKIP (already in results_fairness_over_time\bias_crows_pairs.csv): 2024  google/recurrentgemma-2b  | set force=True to rerun


In [42]:
df_sum  = pd.read_csv(SUMMARY_CSV).sort_values(["year","model_id"]).reset_index(drop=True)
df_bias = pd.read_csv(BIAS_CSV).sort_values(["year","model_id"]).reset_index(drop=True)

df = df_sum.merge(df_bias[["year","model_id","crows_bias"]], on=["year","model_id"], how="left")